In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib, subprocess, sys
from google.colab import userdata
REPO='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Baseline-V1'; RUN_ID='tree_ring_sd35_one_unit_v1'; OFFICIAL='https://github.com/YuxinWenRick/tree-ring-watermark.git'; OFFICIAL_EXACT='3015283d9cf82e90b628f02ad2121bd37408ca9a'; FORCE_RERUN_ALL=False
root=pathlib.Path('/content/cegwm'); subprocess.run(['git','clone',REPO,str(root)],check=True); resolved_exact=subprocess.run(['git','-C',str(root),'rev-parse',f'origin/{BRANCH}'],text=True,capture_output=True,check=True).stdout.strip(); subprocess.run(['git','-C',str(root),'checkout','--detach',resolved_exact],check=True); assert not subprocess.run(['git','-C',str(root),'status','--porcelain'],text=True,capture_output=True,check=True).stdout.strip()
official=pathlib.Path('/content/tree_ring'); subprocess.run(['git','clone',OFFICIAL,str(official)],check=True); subprocess.run(['git','-C',str(official),'checkout','--detach',OFFICIAL_EXACT],check=True); official_head=subprocess.run(['git','-C',str(official),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip(); official_dirty=subprocess.run(['git','-C',str(official),'status','--porcelain'],text=True,capture_output=True,check=True).stdout.strip(); assert official_head==OFFICIAL_EXACT and not official_dirty
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate'],check=True); subprocess.run([sys.executable,'-m','pip','install','-e',str(root)],check=True); child_env=dict(os.environ); child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''; assert child_env['HF_TOKEN']; run_dir=pathlib.Path('/content/drive/MyDrive/CEG-WM/Baseline-V1')/RUN_ID
command=[sys.executable,'-m','cegwm.baselines.external_canary','--method','tree_ring','--run-dir',str(run_dir),'--run-id',RUN_ID,'--project-exact',resolved_exact,'--official-source',str(official)]; command += ['--force-rerun-all'] if FORCE_RERUN_ALL else []; subprocess.run(command,cwd=root,env=child_env,check=True); assert (run_dir/'final_manifest.json').is_file(); print('engineering-only canary; Drive:',run_dir)